# Étape 6 – Cartes : Top 5 des destinations et Top 20 des hôtels

**Objectif** : produire les deux cartes demandées par l'énoncé, **à partir du Data Warehouse (RDS)** :
c'est la preuve que l'entrepôt de données est exploitable par une équipe d'analyse.

**Règles de sélection** :

| Carte | Règle |
|---|---|
| **Top 5 destinations** | Les 5 destinations ayant le meilleur score « beau temps » sur les 7 prochains jours (étape 2) |
| **Top 20 hôtels** | Les **4 meilleurs hôtels de chacune des 5 destinations** (répartition équilibrée) |

Choix pour le Top 20 :
- **répartition équilibrée** (4 × 5) plutôt qu'un classement global : la carte propose un choix d'hôtels pour
  **chaque** destination recommandée (un classement global plaçait 9 hôtels sur 20 à Biarritz) ;
- **une seule ligne par hôtel** : un hôtel proposé pour deux destinations voisines (ex. : l'Hôtel du Palais,
  pour Biarritz et Bayonne) est rattaché à la **plus proche** ;
- hôtels situés à **20 km au plus** de la destination et notés sur **au moins 50 avis** : une note fondée sur
  une dizaine d'avis n'est pas fiable (cohérent avec le tri « note pondérée par le nombre d'avis » utilisé à l'étape 3) ;
- tri par **note**, puis par **nombre d'avis** en cas d'égalité (une note fondée sur plus d'avis est plus fiable).

**Sorties** : deux cartes interactives au format **HTML autonome** dans `outputs/maps/`
(consultables dans n'importe quel navigateur, sans Python).

## 1. Imports, configuration et connexion à RDS

In [1]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL


def find_project_root(start: Path) -> Path:
    """Remonte l'arborescence jusqu'au dossier qui contient requirements.txt (racine du projet)."""
    for folder in [start, *start.parents]:
        if (folder / "requirements.txt").exists():
            return folder
    raise FileNotFoundError("Racine du projet introuvable (requirements.txt absent)")


PROJECT_ROOT = find_project_root(Path.cwd())
MAPS_DIR = PROJECT_ROOT / "outputs" / "maps"
MAPS_DIR.mkdir(parents=True, exist_ok=True)

# override=True : le fichier .env fait foi, même si le kernel a déjà chargé une ancienne valeur
load_dotenv(PROJECT_ROOT / ".env", override=True)

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("RDS_USER"),
    password=os.getenv("RDS_PASSWORD"),
    host=os.getenv("RDS_HOST"),
    port=int(os.getenv("RDS_PORT")),
    database=os.getenv("RDS_DB_NAME"),
)
engine = create_engine(db_url, connect_args={"sslmode": os.getenv("RDS_SSLMODE", "require")})

with engine.connect() as connection:
    print("Connexion OK –", connection.execute(text("SELECT COUNT(*) FROM destinations")).scalar(), "destinations")

Connexion OK – 35 destinations


## 2. Paramètres de sélection

Les règles métier sont regroupées ici : on peut les modifier sans toucher aux requêtes.

In [2]:
TOP_DESTINATIONS = 5          # nombre de destinations retenues
HOTELS_PER_DESTINATION = 4    # hôtels par destination (5 × 4 = 20)
MAX_DISTANCE_KM = 20          # distance maximale hôtel <-> destination
MIN_REVIEW_COUNT = 50         # nombre minimal d'avis pour qu'une note soit jugée fiable

# Une couleur par rang de destination, identique sur les deux cartes
DESTINATION_COLORS = ["#d73027", "#4575b4", "#1a9850", "#984ea3", "#ff7f00"]
OTHER_COLOR = "#b0b0b0"


def png_export_config(filename: str) -> dict:
    """Réglages du bouton 'appareil photo' : nom de fichier propre et image en haute résolution."""
    return {"toImageButtonOptions": {"format": "png", "filename": filename, "scale": 2}}

## 3. Top 5 des destinations (requête SQL sur RDS)

In [3]:
destinations = pd.read_sql(
    text("""
        SELECT city_id, city_name, latitude, longitude, weather_rank, weather_score,
               total_rain_mm, avg_pop, avg_temp_day, rainy_days, weather_period_start, weather_period_end
        FROM destinations
        ORDER BY weather_rank, weather_score DESC
    """),
    engine,
)
destinations["is_top"] = destinations.index < TOP_DESTINATIONS
top_destinations = destinations[destinations["is_top"]]

PERIOD = f"du {destinations['weather_period_start'].min():%d/%m} au {destinations['weather_period_end'].max():%d/%m/%Y}"
print("Période des prévisions :", PERIOD)
top_destinations[["weather_rank", "city_name", "weather_score", "total_rain_mm", "avg_temp_day", "rainy_days"]]

Période des prévisions : du 05/11 au 11/11/2026


,weather_rank,city_name,weather_score,total_rain_mm,avg_temp_day,rainy_days
0,1,Collioure,0.933,1.0,16.8,3
1,2,Bayonne,0.914,5.6,17.0,3
2,3,Carcassonne,0.894,3.0,15.6,2
3,4,Biarritz,0.890,5.6,16.5,4
4,5,Toulouse,0.882,3.7,15.4,2


## 4. Top 20 des hôtels (requête SQL sur RDS)

La requête s'appuie sur des **fonctions de fenêtrage** (`ROW_NUMBER() OVER (PARTITION BY ...)`) :
1. `candidates` : liens hôtel ↔ destination pour les 5 meilleures destinations, à moins de 20 km ;
2. `unique_hotels` : pour chaque hôtel, on ne garde que la destination **la plus proche** (supprime les doublons) ;
3. `ranked` : hôtels ayant au moins 50 avis, classés **au sein de chaque destination** (note, puis nombre d'avis) ;
4. sélection des 4 premiers de chaque destination.

In [4]:
TOP_HOTELS_QUERY = text("""
    WITH top_destinations AS (
        SELECT city_id
        FROM destinations
        ORDER BY weather_rank, weather_score DESC
        LIMIT :top_destinations
    ),
    candidates AS (
        SELECT dh.city_id, dh.hotel_id, dh.distance_km,
               ROW_NUMBER() OVER (PARTITION BY dh.hotel_id ORDER BY dh.distance_km, dh.city_id) AS closest_rank
        FROM destination_hotels AS dh
        JOIN top_destinations AS t ON t.city_id = dh.city_id
        WHERE dh.distance_km <= :max_distance_km
    ),
    unique_hotels AS (
        SELECT city_id, hotel_id, distance_km FROM candidates WHERE closest_rank = 1
    ),
    ranked AS (
        SELECT u.city_id, u.hotel_id, u.distance_km,
               ROW_NUMBER() OVER (
                   PARTITION BY u.city_id
                   ORDER BY h.review_score DESC, h.review_count DESC, h.hotel_id
               ) AS rank_in_destination
        FROM unique_hotels AS u
        JOIN hotels AS h ON h.hotel_id = u.hotel_id
        WHERE h.review_score IS NOT NULL
          AND h.review_count >= :min_review_count
    )
    SELECT d.weather_rank, d.city_name, r.rank_in_destination,
           h.hotel_name, h.review_score, h.review_count, h.star_rating, r.distance_km,
           h.address, h.hotel_city, h.latitude, h.longitude, h.hotel_url
    FROM ranked AS r
    JOIN destinations AS d ON d.city_id = r.city_id
    JOIN hotels       AS h ON h.hotel_id = r.hotel_id
    WHERE r.rank_in_destination <= :hotels_per_destination
    ORDER BY d.weather_rank, r.rank_in_destination
""")

top_hotels = pd.read_sql(
    TOP_HOTELS_QUERY,
    engine,
    params={"top_destinations": TOP_DESTINATIONS, "max_distance_km": MAX_DISTANCE_KM,
            "hotels_per_destination": HOTELS_PER_DESTINATION, "min_review_count": MIN_REVIEW_COUNT},
)

# Contrôles : 20 hôtels, aucun doublon, 4 par destination
assert len(top_hotels) == TOP_DESTINATIONS * HOTELS_PER_DESTINATION, f"{len(top_hotels)} hôtels au lieu de 20"
assert top_hotels["hotel_name"].is_unique, "un hôtel apparaît deux fois"
print("Hôtels par destination :", top_hotels.groupby("city_name").size().to_dict())
top_hotels[["weather_rank", "city_name", "rank_in_destination", "hotel_name", "review_score",
            "review_count", "star_rating", "distance_km"]]

Hôtels par destination : {'Bayonne': 4, 'Biarritz': 4, 'Carcassonne': 4, 'Collioure': 4, 'Toulouse': 4}


,weather_rank,city_name,rank_in_destination,hotel_name,review_score,review_count,star_rating,distance_km
0,1,Collioure,1,Les Roches Brunes,9.3,653,4.0,0.6
1,1,Collioure,2,Intense Boutik-Hôtel,9.3,431,3.0,2.1
2,1,Collioure,3,Bourquin Hotel - Lycée Hôtelier Christian Bour...,9.3,160,NaN,7.8
3,1,Collioure,4,Hôtel La Casa Pairal,9.2,1007,4.0,0.1
4,2,Bayonne,1,Hôtel Villa la Renaissance,8.9,493,4.0,1.4
5,2,Bayonne,2,Hôtel Villa KOEGUI Bayonne,8.8,1271,4.0,0.3
6,2,Bayonne,3,ibis Styles Bayonne,8.6,1101,3.0,0.2
7,2,Bayonne,4,Hôtel des Basses Pyrénées - Bayonne,8.6,906,4.0,0.7
8,3,Carcassonne,1,Hôtel La Bastide Saint Martin,9.3,555,3.0,4.2
9,3,Carcassonne,2,Hôtel Le Parc - La Table de Franck Putelat,9.2,399,5.0,1.6


## 5. Carte 1 – Top 5 des destinations

Les 35 destinations sont affichées : les 5 meilleures en couleur, avec leur rang dans la légende,
les autres en gris, pour situer le classement dans son contexte. (Une légende plutôt que des étiquettes sur la carte :
le moteur cartographique masque les étiquettes qui se chevauchent, comme Bayonne et Biarritz.)

In [5]:
map_data = destinations.assign(
    legend=destinations.apply(
        lambda r: f"{r.weather_rank}. {r.city_name}" if r.is_top else "Autres destinations", axis=1),
    marker_size=destinations["is_top"].map({True: 22, False: 7}),
)
top_labels = [f"{r.weather_rank}. {r.city_name}" for r in top_destinations.itertuples()]
color_map = dict(zip(top_labels, DESTINATION_COLORS)) | {"Autres destinations": OTHER_COLOR}

fig_destinations = px.scatter_map(
    map_data,
    lat="latitude",
    lon="longitude",
    color="legend",
    color_discrete_map=color_map,
    category_orders={"legend": top_labels + ["Autres destinations"]},
    size="marker_size",
    size_max=22,
    hover_name="city_name",
    hover_data={
        "weather_rank": True, "weather_score": ":.3f", "total_rain_mm": ":.1f", "avg_temp_day": ":.1f",
        "rainy_days": True, "legend": False, "marker_size": False, "latitude": False, "longitude": False,
    },
    labels={"weather_rank": "Rang", "weather_score": "Score beau temps", "total_rain_mm": "Pluie sur 7 j (mm)",
            "avg_temp_day": "Température moyenne (°C)", "rainy_days": "Jours de pluie", "legend": "Destination"},
    center={"lat": 46.6, "lon": 2.4},
    zoom=4.6,
    height=700,
    map_style="open-street-map",
    title=f"Kayak – Top 5 des destinations où il fera le plus beau ({PERIOD})",
)
fig_destinations.show(config=png_export_config("top5_destinations"))

## 6. Carte 2 – Top 20 des hôtels (4 par destination)

Une couleur par destination (les mêmes que sur la carte 1) ; le survol affiche la note, le nombre d'avis,
la catégorie et la distance. Zoomer (molette) pour distinguer les hôtels d'une même destination.

In [6]:
fig_hotels = px.scatter_map(
    top_hotels.assign(destination=top_hotels["weather_rank"].astype(str) + ". " + top_hotels["city_name"]),
    lat="latitude",
    lon="longitude",
    color="destination",
    color_discrete_sequence=DESTINATION_COLORS,
    category_orders={"destination": top_labels},
    hover_name="hotel_name",
    hover_data={
        "review_score": True, "review_count": True, "star_rating": True, "distance_km": True,
        "address": True, "destination": False, "latitude": False, "longitude": False,
    },
    labels={"review_score": "Note Booking", "review_count": "Nombre d'avis", "star_rating": "Étoiles",
            "distance_km": "Distance à la destination (km)", "address": "Adresse", "destination": "Destination"},
    center={"lat": top_hotels["latitude"].mean(), "lon": top_hotels["longitude"].mean()},
    zoom=6,
    height=700,
    map_style="open-street-map",
    title=f"Kayak – Top 20 des hôtels dans les 5 meilleures destinations ({PERIOD})",
)
fig_hotels.update_traces(marker={"size": 14})
fig_hotels.show(config=png_export_config("top20_hotels"))

## 7. Export des cartes en HTML autonome

`include_plotlyjs=True` intègre la bibliothèque Plotly dans le fichier : la carte s'ouvre **sans connexion
au serveur Plotly** (seul le fond de carte OpenStreetMap nécessite Internet).
Fichiers plus lourds (~5 Mo), mais utilisables tels quels par le jury ou l'équipe marketing.

Pour une **image PNG** (README, slides) : bouton **appareil photo** en haut à droite de chaque carte
(« Download plot as a png ») : l'image est proposée sous le nom `top5_destinations.png` ou `top20_hotels.png`,
à enregistrer dans `outputs/maps/`.

In [8]:
EXPORTS = {
    "top5_destinations": fig_destinations,
    "top20_hotels": fig_hotels,
}
for name, figure in EXPORTS.items():
    path = MAPS_DIR / f"{name}.html"
    figure.write_html(path, include_plotlyjs=True, full_html=True, config=png_export_config(name))
    filename = path.name
    print(f"{filename:<24} -> {path.stat().st_size / 1_000_000:.1f} Mo")

top5_destinations.html   -> 4.8 Mo
top20_hotels.html        -> 4.8 Mo
